# Description

In this notebook, prepare the data for the experiment with sclability wrt input dimensions.

In [13]:
import numpy as np
import sympy as sp
import h5py

seed = 0
np.random.seed(seed)

def sample_signed_uniform(low=0.5, high=3.0, decimals=2):
    mag = np.random.uniform(low, high)
    sgn = -1.0 if np.random.rand() < 0.5 else 1.0
    return float(np.round(sgn * mag, decimals))

def sample_poly_coeffs(k, quadratic):
    c0 = sample_signed_uniform()
    lin = np.array([sample_signed_uniform() for _ in range(k)], dtype=float)
    quad = None
    if quadratic:
        iu = np.triu_indices(k, k=1)
        quad = np.array([sample_signed_uniform() for _ in range(iu[0].shape[0])], dtype=float)
    return c0, lin, quad

def sample_denominator_with_pole(k, quadratic, domain=5.0):
    x_star = np.random.uniform(-domain, domain, size=(k,)).astype(float)
    lin = np.array([sample_signed_uniform() for _ in range(k)], dtype=float)
    quad = None
    if quadratic:
        iu = np.triu_indices(k, k=1)
        quad = np.array([sample_signed_uniform() for _ in range(iu[0].shape[0])], dtype=float)
        cross = float(np.sum((x_star[iu[0]] * x_star[iu[1]]) * quad))
    else:
        cross = 0.0
    c0 = -float(x_star @ lin + cross)
    c0 = float(np.round(c0, 2))
    return x_star, (c0, lin, quad)

def eval_poly(xk, c0, lin, quad):
    p = c0 + xk @ lin
    if quad is None:
        return p
    k = xk.shape[1]
    iu = np.triu_indices(k, k=1)
    p = p + np.sum((xk[:, iu[0]] * xk[:, iu[1]]) * quad[None, :], axis=1)
    return p

def poly_to_sympy(c0, lin, quad, xs):
    expr = sp.Float(c0)
    for j, a in enumerate(lin):
        expr += sp.Float(float(a)) * xs[j]
    if quad is not None:
        iu = np.triu_indices(len(xs), k=1)
        for t, (i, j) in enumerate(zip(iu[0], iu[1])):
            expr += sp.Float(float(quad[t])) * xs[i] * xs[j]
    return sp.simplify(expr)

def expr_to_sympy(expr):
    k = expr["k"]
    xs = sp.symbols("x0:" + str(k), real=True)
    p = poly_to_sympy(*expr["p"], xs)
    q = poly_to_sympy(*expr["q"], xs)
    y = sp.simplify(p / q)
    return xs, p, q, y

def generate_expression(kind, k=2, pole_domain=5.0):
    if kind not in ("R1", "R2", "R3"):
        raise ValueError("kind must be one of: R1, R2, R3")
    p_quad = kind in ("R2", "R3")
    q_quad = kind == "R3"
    p_coeffs = sample_poly_coeffs(k, quadratic=p_quad)
    pole_x, q_coeffs = sample_denominator_with_pole(k, quadratic=q_quad, domain=pole_domain)
    return {"kind": kind, "k": k, "p": p_coeffs, "q": q_coeffs, "pole_x": pole_x}

def eval_expression(expr, x):
    k = expr["k"]
    xk = x[:, :k]
    p = eval_poly(xk, *expr["p"])
    q = eval_poly(xk, *expr["q"])
    y = p / q
    return y, q

def sample_dataset_train(expr, d, n, delta0=0.05, batch=60000):
    xs = []
    ys = []
    while sum(a.shape[0] for a in xs) < n:
        x = np.random.uniform(-5.0, 5.0, size=(batch, d))
        y, q = eval_expression(expr, x)
        m = np.isfinite(y) & (np.abs(q) >= delta0)
        x = x[m]
        y = y[m]
        if x.shape[0] == 0:
            continue
        xs.append(x)
        ys.append(y)
    X = np.concatenate(xs, axis=0)[:n]
    y = np.concatenate(ys, axis=0)[:n]
    return X, y

def sample_outer_band(n, d, lo=5.0, hi=10.0):
    mag = np.random.uniform(lo, hi, size=(n, d))
    sgn = np.where(np.random.rand(n, d) < 0.5, -1.0, 1.0)
    return sgn * mag

def sample_dataset_test_outer(expr, d, n, delta0=0.05, batch=90000):
    xs = []
    ys = []
    while sum(a.shape[0] for a in xs) < n:
        x = sample_outer_band(batch, d, lo=5.0, hi=10.0)
        y, q = eval_expression(expr, x)
        m = np.isfinite(y) & (np.abs(q) >= delta0)
        x = x[m]
        y = y[m]
        if x.shape[0] == 0:
            continue
        xs.append(x)
        ys.append(y)
    X = np.concatenate(xs, axis=0)[:n]
    y = np.concatenate(ys, axis=0)[:n]
    return X, y

def write_str_dataset(g, name, s):
    dt = h5py.string_dtype(encoding="utf-8")
    if name in g:
        del g[name]
    g.create_dataset(name, data=np.array(s, dtype=dt))

def write_array(g, name, arr, compression="gzip", compression_opts=4):
    if name in g:
        del g[name]
    g.create_dataset(
        name,
        data=arr,
        compression=compression,
        compression_opts=compression_opts,
        shuffle=True,
        chunks=True,
    )

if __name__ == "__main__":
    k = 2
    delta0 = 0.01
    n_train = 10000
    n_test = 10000
    d_list = [2, 4, 8, 16, 32, 64]
    out_path = "scalability_experiment_data.h5"

    exprs = [
        generate_expression("R1", k=k, pole_domain=5.0),
        generate_expression("R2", k=k, pole_domain=5.0),
        generate_expression("R3", k=k, pole_domain=5.0),
    ]

    with h5py.File(out_path, "w") as f:
        f.attrs["k"] = k
        f.attrs["delta0"] = float(delta0)
        f.attrs["train_domain_min"] = -5.0
        f.attrs["train_domain_max"] = 5.0
        f.attrs["test_domain_min_abs"] = 5.0
        f.attrs["test_domain_max_abs"] = 10.0
        f.attrs["n_train"] = int(n_train)
        f.attrs["n_test"] = int(n_test)
        f.attrs["d_list"] = np.array(d_list, dtype=np.int32)

        g_expr = f.require_group("expressions")
        g_data = f.require_group("data")

        for expr in exprs:
            kind = expr["kind"]
            _, p_sym, q_sym, y_sym = expr_to_sympy(expr)

            ge = g_expr.require_group(kind)
            ge.attrs["kind"] = kind
            ge.attrs["k"] = int(expr["k"])
            write_array(ge, "pole_x", np.asarray(expr["pole_x"], dtype=np.float32))

            p_c0, p_lin, p_quad = expr["p"]
            q_c0, q_lin, q_quad = expr["q"]

            ge.attrs["p_c0"] = float(p_c0)
            write_array(ge, "p_lin", np.asarray(p_lin, dtype=np.float32))
            write_array(ge, "p_quad", np.zeros((0,), dtype=np.float32) if p_quad is None else np.asarray(p_quad, dtype=np.float32))

            ge.attrs["q_c0"] = float(q_c0)
            write_array(ge, "q_lin", np.asarray(q_lin, dtype=np.float32))
            write_array(ge, "q_quad", np.zeros((0,), dtype=np.float32) if q_quad is None else np.asarray(q_quad, dtype=np.float32))

            write_str_dataset(ge, "sympy_P_srepr", sp.srepr(p_sym))
            write_str_dataset(ge, "sympy_Q_srepr", sp.srepr(q_sym))
            write_str_dataset(ge, "sympy_y_srepr", sp.srepr(y_sym))
            write_str_dataset(ge, "sympy_P_str", str(p_sym))
            write_str_dataset(ge, "sympy_Q_str", str(q_sym))
            write_str_dataset(ge, "sympy_y_str", str(y_sym))

            gd_kind = g_data.require_group(kind)
            for d in d_list:
                X_train, y_train = sample_dataset_train(expr, d=d, n=n_train, delta0=delta0)
                X_test, y_test = sample_dataset_test_outer(expr, d=d, n=n_test, delta0=delta0)

                X_train = X_train.astype(np.float32, copy=False)
                y_train = y_train.astype(np.float32, copy=False)
                X_test = X_test.astype(np.float32, copy=False)
                y_test = y_test.astype(np.float32, copy=False)

                gd = gd_kind.require_group(f"d{d}")
                gt = gd.require_group("train")
                gs = gd.require_group("test")

                write_array(gt, "X", X_train)
                write_array(gt, "y", y_train)
                write_array(gs, "X", X_test)
                write_array(gs, "y", y_test)

    print("Saved:", out_path)


Saved: scalability_experiment_data.h5
